# LLM Adaptation and Inference Lab

Adapt an autoregressive model, compare retrieval and tuning, and quantify serving trade-offs.

Run the lecture cells in sequence with Python 3.10 or later. The default exercises use bundled data and require no model account.


## Practice case · A smaller model for ticket routing

Attiya supports a routing feature with a tight compute budget. The model must learn a stable output convention and must use current guidance when a team moves. She compares a frozen model, a small adaptation, and a retrieval baseline, then checks how precision choices affect the service.

The workplace scenario is fictional.


## Establish and adapt baselines


### Lecture 01 · Autoregressive evaluation and retrieval baseline

Establish frozen and retrieval-augmented baselines on a held-out task.

#### Recognize a retrieval problem before you train

A model chooses the wrong team for a newly reorganized unit. On older cases its label format is already correct.

Use a retrieval-assisted baseline for the current directory. Measure it alongside the frozen model on the same held-out requests. Reserve adaptation for a recurring behavior where updating facts alone cannot solve the problem.

#### How it works

An autoregressive model assigns probabilities to next tokens. Compare task behavior and log loss on the same cases. A retrieval baseline can supply new facts without changing model parameters, which is useful when facts change frequently.

##### Separate knowledge from behavior before changing weights

A frozen model is a baseline for the task as initially specified. A retrieval-assisted version can add current evidence without modifying that model. An adapted version changes behavior learned from examples. Compare all three on identical held-out inputs so a routing convention improvement is not confused with adding the latest directory.

An autoregressive model predicts a distribution for the next token given earlier tokens. For a routing task, inspect whether the expected label is a single token for the local tokenizer. If tokenization differs, score complete labels consistently. Measure both task outcomes and how confidently the model assigns probability to the expected label.

Include an unfamiliar category and a case that should go to staff. A model that labels routine requests more consistently may still fail when outside its evidence. Keep the simpler retrieval or rule baseline and identify what the extra training is meant to improve before spending compute.

#### Worked example

A policy task needs unfamiliar contact names. A retrieval hint gives the correct evidence without modifying the base model.

#### Try it

Run the bundled base and retrieval baseline on the held-out cases. Inspect where retrieval has the relevant evidence.


In [ ]:
from dataclasses import dataclass
from math import exp, log


@dataclass(frozen=True)
class Example:
    text: str
    label: int


labels = ("open", "review", "other")
train = (
    Example("routine facilities task", 0), Example("routine supplies request", 0),
    Example("restricted staff question", 1), Example("restricted policy change", 1),
)
test = (
    Example("routine schedule change", 0), Example("restricted legal request", 1),
    Example("routine delivery update", 0), Example("restricted vendor details", 1),
)
assert not {row.text for row in train} & {row.text for row in test}


def features(text):
    first, *_ = text.split()
    return (float(first == "routine"), float(first == "restricted"))


def probabilities(logits):
    maximum = max(logits)
    weights = [exp(value - maximum) for value in logits]
    total = sum(weights)
    return [weight / total for weight in weights]


def frozen(text):
    return (0.1, 0.0, 0.2)


guides = {"routine": 0}  # This small source has no restricted-policy guidance.


def retrieval(text):
    hint = guides.get(text.split()[0])
    return tuple(value + (2 if index == hint else 0) for index, value in enumerate(frozen(text)))


def metrics(model, cases):
    outputs = [probabilities(model(row.text)) for row in cases]
    correct = sum(max(range(len(dist)), key=dist.__getitem__) == row.label for dist, row in zip(outputs, cases))
    loss = -sum(log(dist[row.label]) for dist, row in zip(outputs, cases)) / len(cases)
    return {"accuracy": correct / len(cases), "log_loss": loss}


print("Frozen", metrics(frozen, test), "retrieval", metrics(retrieval, test))


**Compare your result**

Report both baselines and preserve the held-out examples while deciding what to adapt.


### Lecture 02 · Train and evaluate a low-rank adapter

Train a small adapter over frozen parameters and evaluate held-out next-token predictions.

#### Inspect held-out behavior after adapting

A low-rank correction learns the labels in training. The evaluation set also contains unfamiliar phrasing that still needs human follow-up.

Compare loss and routing accuracy on data held out before training. Verify that base parameters stayed fixed. Compare against retrieval because a change in adaptation accuracy alone cannot tell you which system to ship.

#### How it works

A low-rank adapter writes a correction as the product of two small matrices. The base weights stay frozen. Optimize on training loss and compare held-out metrics using the same tokenizer, cases, and scoring approach.

##### Tune a small correction and inspect what transfers

A low-rank adapter represents a weight adjustment as two smaller factors. During training, keep the base fixed and optimize the factors on the training split. This reduces the parameters you need to update, while the base model still determines much of the representation available to the correction.

Compare training loss with a held-out task score. Fitting familiar labels can improve likelihood without improving what users receive. Inspect whether the model learned a stable convention or simply memorized the examples. Include valid evidence, contradictory evidence, and a realistic request that should be handed off.

Record adapter shape, base revision, data split, and optimizer settings. If the adapted configuration improves only easy requests or regresses critical slices, test a narrower route or retrieval baseline. The small workbook illustrates the mechanics of adaptation. Its result does not establish the expected gain for another model or workload.

#### Worked example

A repeated output convention can be learned from examples. A recently changed phone number is better supplied by current evidence.

#### Try it

Train the bundled adapter and compare held-out results. Change the rank or training budget and record the quality change.


In [ ]:
# Rank one correction: A maps two input features to one number; B maps that to three logits.
a = [0.15, -0.15]
b = [0.0, 0.0, 0.0]


def forward(text, left, right):
    hidden = sum(weight * feature for weight, feature in zip(left, features(text)))
    return [value + hidden * weight for value, weight in zip(frozen(text), right)]


def adapted(text):
    return forward(text, a, b)


base_before = [frozen(row.text) for row in test]
for _ in range(100):
    for row in train:
        x = features(row.text)
        hidden = sum(weight * feature for weight, feature in zip(a, x))
        residual = [p - int(i == row.label) for i, p in enumerate(probabilities(adapted(row.text)))]
        grad_hidden = sum(error * weight for error, weight in zip(residual, b))
        a = [weight - 0.08 * grad_hidden * feature for weight, feature in zip(a, x)]
        b = [weight - 0.08 * error * hidden for weight, error in zip(b, residual)]

assert [frozen(row.text) for row in test] == base_before
assert metrics(adapted, test)["log_loss"] < metrics(frozen, test)["log_loss"]
assert metrics(adapted, test)["accuracy"] == 1
print("Adapted", metrics(adapted, test))


**Compare your result**

The frozen base stays unchanged. Report when adaptation improves behavior and identify any regression relative to retrieval.


## Compress and transfer


### Lecture 03 · Quantization and inference measurement

Quantize a learned correction and compare held-out quality, payload bytes, and latency.

#### Memory savings and latency savings are different observations

A lower-precision correction is smaller to store, but unpacking it for each request takes extra work in a local implementation.

Measure payload size including scale metadata and compare repeated timings on the same hardware. Report quality drift separately from memory and speed. Name the parts of inference the timing actually includes.

#### How it works

Symmetric quantization maps floating-point values to an integer range using a scale. Keep model quality and numeric drift separate from performance. A smaller payload can reduce transfers, while dequantization can add work on hardware without suitable kernels.

##### Measure precision trade-offs on the path that runs

Quantization stores values with lower precision. A typical integer scheme pairs compact values with scales that approximate the original values when used. Payload bytes include those scales and any metadata. Also report whether the measurement covers the adapter, the base model, or the whole deployment.

Lower storage cost is independent of faster inference. Unpacking or converting values can cost more time in a local implementation. A production runtime with suitable kernels might behave differently. Repeat the same workload on the same machine, record warm-up and measurement conditions, and report latency separately from quality.

Check both task behavior and numerical drift. Small changes in scores can flip a decision near a threshold even when average performance barely moves. Use cases near the edge and actual routing outcomes to decide whether a compression setting is acceptable.

#### Worked example

An int8 correction uses fewer payload bytes. In a small Python implementation, unpacking can offset any latency benefit.

#### Try it

Run repeated measurements before and after quantizing. Record scale overhead and name the hardware.


In [ ]:
from array import array
from statistics import median
from time import perf_counter


def quantize(values):
    scale = max(abs(value) for value in values) / 127 or 1
    return array("b", (round(value / scale) for value in values)), array("f", [scale])


# Pack the whole correction with one scale. The scale is included in the byte count.
float_payload = array("f", a + b)
int_payload, scale = quantize(float_payload)


def compressed(text):
    unpacked = [item * scale[0] for item in int_payload]
    return forward(text, unpacked[:len(a)], unpacked[len(a):])


def timed(model, count=300):
    values = []
    for _ in range(count):
        start = perf_counter()
        model(test[0].text)
        values.append((perf_counter() - start) * 1e6)
    return median(values)


compressed_bytes = len(int_payload.tobytes()) + len(scale.tobytes())
assert compressed_bytes < len(float_payload.tobytes())
assert abs(metrics(adapted, test)["log_loss"] - metrics(compressed, test)["log_loss"]) < 0.05
print("float bytes", len(float_payload.tobytes()), "compressed bytes", compressed_bytes,
      "median µs float/int8", timed(adapted), timed(compressed))


**Compare your result**

Report actual bytes for the matrix payload and the measured timing distribution. Clarify which model components the demonstration compresses.


### Lecture 04 · Transfer to an open-weight checkpoint

Run the same train and holdout comparison against a compatible local transformer checkpoint.

#### Transfer the experiment without transferring the claim

The small fixture improves, while a transformer checkpoint uses another tokenizer and may split a routing label differently.

Confirm the output label representation and compare frozen and adapted versions on the same checkpoint and cases. A lab fixture teaches the measurement method. Production quality still needs representative traffic, a tested fallback, and proportionate rollout.

#### How it works

An open-weight model adds tokenization, pretrained weights, and hardware-dependent serving. Freeze the checkpoint and train a low-rank output correction using synthetic labels. Use isolated train and test cases, and compare whether the change improves the intended task enough to justify deployment.

##### Transfer the protocol as you change the model

A checkpoint experiment should record its model identity, tokenizer, local precision, label format, and hardware. Freeze the task and held-out cases first. Once the checkpoint is loaded, measure the frozen baseline before training the correction so the comparison remains interpretable.

Inspect one input and its label encoding. If an expected label spans several tokens, the score you get from looking at one token does not describe that label. Match the score to the task, and ensure the fallback remains in the evaluation. Report both result quality and the cost of reaching it.

If the optional transformer checkpoint does not improve, preserve that result. Compare it with the simple fixture and identify which assumption stopped transferring. The exercise is about producing evidence for a deployment decision, including a choice to keep the simpler baseline or restore the known-good configuration.

#### Worked example

A small local causal LM can be adapted to choose two response labels. Changes in label tokenization affect what the classifier can reliably measure.

#### Try it

Run the default offline comparison. For the optional transfer, configure a compatible local checkpoint as described in the workbook and compare held-out label probabilities.


In [ ]:
import os


def checkpoint_experiment(path):
    # Install torch and transformers in the same Python environment before opting in.
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    torch.manual_seed(4)
    tokenizer = AutoTokenizer.from_pretrained(path, local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(path, local_files_only=True).eval()
    model.requires_grad_(False)
    token_ids = [tokenizer.encode(" " + label, add_special_tokens=False) for label in labels[:2]]
    if any(len(ids) != 1 for ids in token_ids) or token_ids[0] == token_ids[1]:
        raise ValueError("Choose a checkpoint with distinct single tokens for ' open' and ' review'.")

    def collect(rows):
        outputs = []
        with torch.no_grad():
            for row in rows:
                prompt = f"Ticket: {row.text}\nRoute:"
                result = model(**tokenizer(prompt, return_tensors="pt"), output_hidden_states=True)
                hidden = result.hidden_states[-1][0, -1]
                logits = result.logits[0, -1, [ids[0] for ids in token_ids]]
                outputs.append((hidden.float(), logits.float()))
        x, base = zip(*outputs)
        return torch.stack(x), torch.stack(base), torch.tensor([row.label for row in rows])

    x_train, base_train, y_train = collect(train)
    x_test, base_test, y_test = collect(test)
    left = torch.nn.Linear(x_train.shape[1], 2, bias=False)
    right = torch.nn.Linear(2, 2, bias=False)
    torch.nn.init.zeros_(right.weight)
    optimizer = torch.optim.AdamW([*left.parameters(), *right.parameters()], lr=0.02)
    for _ in range(40):
        optimizer.zero_grad()
        loss = torch.nn.functional.cross_entropy(base_train + right(left(x_train)), y_train)
        loss.backward()
        optimizer.step()
    with torch.no_grad():
        after = base_test + right(left(x_test))
        comparison = {}
        for name, logits in (("frozen", base_test), ("adapted", after)):
            comparison[name] = {
                "log_loss": torch.nn.functional.cross_entropy(logits, y_test).item(),
                "accuracy": (logits.argmax(dim=-1) == y_test).float().mean().item(),
            }
        assert torch.isfinite(after).all() and torch.count_nonzero(right.weight) > 0
        assert not any(parameter.grad is not None for parameter in model.parameters())
        print("Checkpoint", path, "held-out two-label comparison", comparison)


checkpoint = os.getenv("AIEBOK_LOCAL_CHECKPOINT")
if checkpoint:
    checkpoint_experiment(checkpoint)
else:
    print("Offline exercise passed. Install torch and transformers and set AIEBOK_LOCAL_CHECKPOINT to a local causal LM directory to run the optional comparison.")


**Compare your result**

Record the checkpoint and tokenizer versions. Keep the default runnable result separate from the checkpoint experiment and compare held-out evidence before considering release.


## Course project

Compare a frozen baseline, a retrieval-augmented baseline, and an adapted model. Report held-out task results and quality after reducing precision. Use the optional local checkpoint lab for foundation-model transfer.

Keep train and held-out data separate. Measure inference quality, stored parameter memory, and repeated latency. Publish the model identity, hardware, precision, and a keep or revert decision.
